In [3]:
!pip install tensorflow

/opt/conda/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


In [4]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

data_dir = "/kaggle/input/persian-license-plates/"

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    data_dir,
    target_size=(64,64),
    batch_size=32,
    color_mode="grayscale",
    class_mode="categorical"
)

validation_generator = test_datagen.flow_from_directory(
    data_dir,
    target_size=(64,64),
    batch_size=32,
    color_mode="grayscale",
    class_mode="categorical"
)


Found 83844 images belonging to 28 classes.
Found 83844 images belonging to 28 classes.


In [5]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense


image_shape = (64,64,1)
num_classes = train_generator.num_classes

model = Sequential()
model.add(Conv2D(32, (3, 3), activation="relu", input_shape=image_shape))
model.add(MaxPooling2D((2,2)))
model.add(Conv2D(64, (3,3), activation="relu"))
model.add(MaxPooling2D((2,2)))
model.add(Flatten())
model.add(Dense(64, activation="relu"))
model.add(Dense(num_classes, activation="softmax"))

model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=['accuracy'])

In [6]:
epochs = 10

steps_per_epoch = train_generator.samples // train_generator.batch_size

history = model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=epochs,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // validation_generator.batch_size
)

Epoch 1/10
2620/2620 [==============================] - 509s 194ms/step - loss: 1.3262 - accuracy: 0.6053 - val_loss: 0.4141 - val_accuracy: 0.8895
Epoch 2/10
2620/2620 [==============================] - 407s 156ms/step - loss: 0.6905 - accuracy: 0.7882 - val_loss: 0.3181 - val_accuracy: 0.9051
Epoch 3/10
2620/2620 [==============================] - 382s 146ms/step - loss: 0.5382 - accuracy: 0.8336 - val_loss: 0.2784 - val_accuracy: 0.9115
Epoch 4/10
2620/2620 [==============================] - 409s 156ms/step - loss: 0.4470 - accuracy: 0.8649 - val_loss: 0.2359 - val_accuracy: 0.9271
Epoch 5/10
2620/2620 [==============================] - 412s 157ms/step - loss: 0.3867 - accuracy: 0.8822 - val_loss: 0.1876 - val_accuracy: 0.9447
Epoch 6/10
2620/2620 [==============================] - 415s 158ms/step - loss: 0.3445 - accuracy: 0.8961 - val_loss: 0.1563 - val_accuracy: 0.9547
Epoch 7/10
2620/2620 [==============================] - 411s 157ms/step - loss: 0.3151 - accuracy: 0.9045 - val_

In [7]:
# Save the weights
model.save_weights("model_weights.h5")